In [49]:
from langgraph.graph import StateGraph,START,END
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv
from typing import TypedDict,Annotated
from pydantic import BaseModel,Field
import os,operator
load_dotenv()

True

In [50]:
model=ChatGoogleGenerativeAI(
    model='gemini-3.5-flash-lite',
    google_api_key=os.getenv("GEMINI_API_KEY_4")
)

In [51]:
class EvalSchema(BaseModel):
    feedback:str = Field(description="Detaield Feedback for the essay")
    score : int = Field(description="Score out of 10",ge=0,le=10)

In [52]:
structured_model=model.with_structured_output(EvalSchema)

In [53]:
essay = """
Artificial Intelligence (AI) is the defining technology of the twenty-first century, fundamentally altering how humanity works, communicates, and solves problems. At its core, AI refers to the simulation of human intelligence by computer systems, enabling machines to learn from data, reason through complex tasks, and adapt to new inputs. Once confined to the realms of science fiction, AI has rapidly transitioned into an omnipresent reality, driving innovation across global industries while simultaneously presenting profound ethical and existential questions.
The evolution of AI is broadly categorized into two types: Narrow AI and General AI. Narrow AI, or Weak AI, is designed to perform specific tasks with high proficiency. This is the technology currently integrated into daily life, powering facial recognition, financial fraud detection, autonomous vehicles, and conversational algorithms like virtual assistants. Conversely, Artificial General Intelligence (AGI), or Strong AI, remains a theoretical concept. AGI represents a system that possesses human-like cognitive abilities across a broad spectrum of disciplines, capable of abstract thinking, emotional comprehension, and autonomous problem-solving. While AGI remains a distant milestone, the rapid advancement of Narrow AI has already triggered a paradigm shift in human capability.
In terms of societal impact, AI serves as an unprecedented catalyst for efficiency and discovery. In healthcare, machine learning algorithms analyze medical imagery with greater speed and accuracy than human eyes, expediting cancer detection and pioneering the synthesis of new pharmaceuticals. In education, AI platforms offer personalized tutoring tailored to individual learning speeds, democratizing quality education globally. Furthermore, in the fight against climate change, AI optimizes energy grids, tracks deforestation in real-time, and models complex meteorological data to predict severe weather patterns. By automating routine intellectual and physical tasks, AI frees human capital to focus on creative, empathetic, and strategic endeavors.
However, the rapid integration of AI introduces severe challenges that require careful governance. The most immediate concern is economic disruption. As automation expands from manufacturing into white-collar professions, millions of jobs risk displacement, necessitating comprehensive labor retraining programs. Additionally, AI systems are vulnerable to systemic bias. Because machine learning models train on historical data, they often replicate and amplify societal prejudices, leading to discriminatory outcomes in algorithmic hiring, credit scoring, and criminal justice systems. Furthermore, the rise of deepfakes and automated misinformation poses an unprecedented threat to democratic processes and digital trust.
In conclusion, Artificial Intelligence represents a dual-edged sword of immense capability and risk. It is neither inherently benevolent nor malevolent; rather, its impact depends entirely on human guidance, ethical frameworks, and regulatory oversight. To maximize the benefits of this technological revolution, international communities must collaborate to build transparent, accountable, and unbiased AI systems. Managed wisely, AI will not replace human potential, but amplify it, guiding civilization toward a more efficient, informed, and prosperous future.
"""

In [54]:
prompt=f"Evaluate the language quality of the provided essay and provide a feedback and assign a score out of 10\n {essay}"

In [55]:
structured_model.invoke(prompt).score

10

In [56]:
class NEB(TypedDict):
    essay:str
    language_feedback:str
    analysis_feedback:str
    clarity_feedback:str
    overall_feedback:str
    individual_scores:Annotated[list[int],operator.add]
    avg_scre:float

In [57]:
def evalLanguage(state:NEB):
    prompt=f"Evaluate the language quality of the provided essay and provide a feedback and assign a score out of 10\n {state['essay']}"
    res=structured_model.invoke(prompt)
    return {'language_feedback':res.feedback,'individual_scores':[res.score]}

In [58]:
def evalThought(state:NEB):
    prompt=f"Evaluate the clarity of thought of the provided essay and provide a feedback and assign a score out of 10\n {state['essay']}"
    res=structured_model.invoke(prompt)
    return {'clarity_feedback':res.feedback,'individual_scores':[res.score]}

In [59]:
def evalAnalysis(state:NEB):
    prompt=f"Evaluate the clarity of thought of the provided essay and provide a feedback and assign a score out of 10\n {state['essay']}"
    res=structured_model.invoke(prompt)
    return {'analysis_feedback':res.feedback,'individual_scores':[res.score]}

In [60]:
def evalFinal(state:NEB):
    prompt=f"based on the following feedbacks create a summarized feedback \n language feedback - {state['language_feedback']} \n analysis feedback - {state['analysis_feedback']} \n clarity of thought feedback = {state['clarity_feedback']}"
    res=model.invoke(prompt).content[0]['text']
    avg_scre=sum(state['individual_scores'])/len(state['individual_scores'])
    return {'overall_feedback':res,'avg_scre':avg_scre}
    

In [61]:
graph = StateGraph(NEB)

graph.add_node('evalLanguage',evalLanguage)
graph.add_node('evalAnalysis',evalAnalysis)
graph.add_node('evalThought',evalThought)
graph.add_node('evalFinal',evalFinal)

In [62]:
graph.add_edge(START,'evalLanguage')
graph.add_edge(START,'evalAnalysis')
graph.add_edge(START,'evalThought')

graph.add_edge('evalLanguage','evalFinal')
graph.add_edge('evalAnalysis','evalFinal')
graph.add_edge('evalThought','evalFinal')

graph.add_edge('evalFinal',END)


In [63]:
workflow=graph.compile()

In [64]:
initial_state={
    'essay':essay
}

workflow.invoke(initial_state)

{'essay': '\nArtificial Intelligence (AI) is the defining technology of the twenty-first century, fundamentally altering how humanity works, communicates, and solves problems. At its core, AI refers to the simulation of human intelligence by computer systems, enabling machines to learn from data, reason through complex tasks, and adapt to new inputs. Once confined to the realms of science fiction, AI has rapidly transitioned into an omnipresent reality, driving innovation across global industries while simultaneously presenting profound ethical and existential questions.\nThe evolution of AI is broadly categorized into two types: Narrow AI and General AI. Narrow AI, or Weak AI, is designed to perform specific tasks with high proficiency. This is the technology currently integrated into daily life, powering facial recognition, financial fraud detection, autonomous vehicles, and conversational algorithms like virtual assistants. Conversely, Artificial General Intelligence (AGI), or Stron